<center>

# **import Libraries**

In [2]:
# basic imports
import os
import json

from groq import Groq
from langchain.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain.chains import LLMChain
from langgraph.graph import StateGraph, END


In [5]:
# Cell 1: Imports & Setup
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
import json
from collections import deque

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=api_key
)

In [7]:
# Cell 3: Simple conversation memory (last N turns)
class ConversationMemory:
    def __init__(self, max_turns=3):

        self.history = deque(maxlen=max_turns)  # (user_msg, bot_response)

    def add(self, user_msg, bot_response):
        self.history.append({"user": user_msg, "bot": bot_response})

    def get_context(self):
        if not self.history:
            return ""
        lines = []
        for turn in self.history:
            lines.append(f"User: {turn['user']}")
            lines.append(f"Bot: {turn['bot']}")
        return "\n".join(lines)

    def is_empty(self):
        return len(self.history) == 0

memory = ConversationMemory(max_turns=3)

In [8]:
# Cell 4: Prompts

# First turn prompt (no history)
FIRST_TURN_PROMPT = """You are an intent classifier for a mental health chatbot.

Classify the user's message into ONE of these intents:
- greeting
- goodbye  
- gratitude
- asking_mental_health_question
- out_of_scope

Also check if the message is related to mental health topics (anxiety, depression, stress, crisis, emotions, therapy, wellbeing, etc.)

Respond ONLY with valid JSON, no markdown, no explanation:
{{"intent": "<intent>", "is_mental_health_related": <true/false>}}

User message: "{user_message}"
"""

# =================================================================================================================================

# Follow-up turn prompt (with history) - handles "tell me more", "what do you mean", etc.
FOLLOWUP_TURN_PROMPT = """You are an intent classifier for a mental health chatbot.

Here is the recent conversation:
{history}

Now the user says: "{user_message}"

Consider the FULL context. If the user is continuing a mental health topic (e.g. "tell me more", "explain that", "what do you mean"), classify it as asking_mental_health_question.

Classify into ONE intent:
- greeting
- goodbye
- gratitude
- asking_mental_health_question
- out_of_scope

Also check: is this turn related to mental health (directly or via context)?

Respond ONLY with valid JSON, no markdown, no explanation:
{{"intent": "<intent>", "is_mental_health_related": <true/false>}}
"""

In [9]:
# Cell 5: Classify intent — single LLM call, low latency

def classify_intent(user_message: str, memory: ConversationMemory) -> dict:
    """
    Returns: {"intent": str, "is_mental_health_related": bool}
    """
    if memory.is_empty():
        prompt = FIRST_TURN_PROMPT.format(user_message=user_message)
    else:
        prompt = FOLLOWUP_TURN_PROMPT.format(
            history=memory.get_context(),
            user_message=user_message
        )

    response = llm.invoke([HumanMessage(content=prompt)])
    
    try:
        result = json.loads(response.content.strip())
    except json.JSONDecodeError:
        # fallback: safe default
        result = {"intent": "out_of_scope", "is_mental_health_related": False}
    
    return result

In [10]:
# Cell 6: Router — connects intent to the right action

def route(user_message: str, emotion: str, language: str, memory: ConversationMemory) -> dict:
    """
    emotion: output from Module 2
    language: output from Module 1
    """
    classification = classify_intent(user_message, memory)
    intent = classification["intent"]
    is_mh = classification["is_mental_health_related"]
    
    # Override: if LLM said out_of_scope but context is MH → trust context
    if intent == "out_of_scope" and is_mh:
        intent = "asking_mental_health_question"
    
    routes = {
        "greeting":                    "direct_reply",
        "goodbye":                     "direct_reply",
        "gratitude":                   "direct_reply",
        "asking_mental_health_question": "rag_pipeline",
        "out_of_scope":                "polite_refusal",
    }
    
    return {
        "intent": intent,
        "action": routes.get(intent, "polite_refusal"),
        "emotion": emotion,
        "language": language,
        "is_mental_health_related": is_mh
    }

In [12]:
# Cell 7: Test the module

test_cases = [
    "Hello, how are you?",
    "I've been feeling really anxious lately",
    "Tell me more about that",
    "What's the weather today?",
    "Thank you so much!",
    "I feel like I can't go on anymore",
]

for msg in test_cases:
    result = route(
        user_message=msg,
        emotion="neutral",   # coming from Module 2
        language="en",       # coming from Module 1
        memory=memory
    )
    print(f"MSG: {msg!r}")
    print(f"  → intent={result['intent']} | action={result['action']} | MH={result['is_mental_health_related']}\n")
    
    # Simulate saving to memory
    memory.add(msg, "[bot response here]")

MSG: 'Hello, how are you?'
  → intent=greeting | action=direct_reply | MH=False

MSG: "I've been feeling really anxious lately"
  → intent=asking_mental_health_question | action=rag_pipeline | MH=True

MSG: 'Tell me more about that'
  → intent=asking_mental_health_question | action=rag_pipeline | MH=True

MSG: "What's the weather today?"
  → intent=out_of_scope | action=polite_refusal | MH=False

MSG: 'Thank you so much!'
  → intent=gratitude | action=direct_reply | MH=False

MSG: "I feel like I can't go on anymore"
  → intent=asking_mental_health_question | action=rag_pipeline | MH=True



<center>

# **Direct response generator**

In [13]:
# These handle greeting, goodbye, gratitude, out_of_scope

DIRECT_RESPONSE_PROMPT = """You are a warm, empathetic mental health support chatbot assistant.

The user's intent is: {intent}
The user's emotion (from classifier): {emotion}
The user's language: {language}

User message: "{user_message}"

Respond naturally and warmly in the SAME language as the user.
- greeting → welcome them, ask how they're feeling
- goodbye  → warm farewell, remind them support is always here
- gratitude → acknowledge warmly, you're happy to help
- out_of_scope → politely say you specialize in mental health support only

Keep it short: 1-3 sentences max.
"""

def generate_direct_response(intent: str, user_message: str,
                              emotion: str, language: str) -> str:
    prompt = DIRECT_RESPONSE_PROMPT.format(
        intent=intent,
        emotion=emotion,
        language=language,
        user_message=user_message
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

<center>

# **Full Module + Test**

In [15]:
# Cell 9: Full Module 3 — classify + respond (except MH → will call RAG later)

def module3_pipeline(user_message: str, emotion: str,
                     language: str, memory: ConversationMemory) -> dict:
    """
    Returns dict with:
      - intent, action, response (if direct)
      - or action='rag_pipeline' so the integration layer calls Module 4
    """
    routing = route(user_message, emotion, language, memory)
    intent  = routing["intent"]
    action  = routing["action"]

    if action == "rag_pipeline":
        # Module 4 will handle this — just return the routing info
        result = {**routing, "response": None, "needs_rag": True}
    else:
        # Handle directly here
        reply = generate_direct_response(intent, user_message, emotion, language)
        result = {**routing, "response": reply, "needs_rag": False}

    # Save to memory (response placeholder for rag case — updated later)
    memory.add(user_message, result["response"] or "[rag_response]")
    return result

# ---- Quick test ----
memory2 = ConversationMemory()
tests = [
    ("Hello there!", "neutral", "en"),
    ("I've been feeling so depressed", "sadness", "en"),
    ("What's 2+2?", "neutral", "en"),
    ("Thank you!", "neutral", "en"),
]
for msg, emo, lang in tests:
    out = module3_pipeline(msg, emo, lang, memory2)
    print(f"[{out['intent']}] >> needs_rag={out['needs_rag']}")
    if out['response']:
        print(f"  Response: {out['response']}")
    print()

[greeting] >> needs_rag=False
  Response: Hello there! It's lovely to connect with you. How are you feeling today?

[asking_mental_health_question] >> needs_rag=True

[out_of_scope] >> needs_rag=False
  Response: I'm happy to chat with you, but I want to make sure I'm here to support you in the best way possible. Math questions aren't really my area of expertise, but I'm here to listen and help with anything on your mind that's related to your mental health and well-being. Would you like to talk about something else?

[gratitude] >> needs_rag=False
  Response: You're welcome, it was my pleasure to help. I'm glad I could be of assistance to you.

